# Week 82, Day 5: Design interview practice II

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w082d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
messages = [
    {"from": "ada", "text": "Are you free at 3?", "client_time": 1000.0},
    {"from": "bob", "text": "Yes, see you then", "client_time": 999.2},   # Bob's phone clock is slow
]
for m in sorted(messages, key=lambda m: m["client_time"]):
    print(m["from"], ":", m["text"])

> **Key idea.** Clarify (groups, devices, history, media, receipts), estimate (here 23,000 messages a second, 400 GB a day, hundreds of connection servers), then design: WebSocket gateways hold device connections, a session registry maps devices to gateways, the chat service stores each message before fanning it out, and offline devices get push notifications. Order by per-conversation sequence numbers assigned atomically on arrival, never by device clocks. Make sends idempotent with client message ids. Sync with per-conversation cursors (the highest sequence received): the server returns everything after each cursor. Receipts are per-member delivered and read cursors; delivered or read by all is the minimum over the other members; unread counts are other people's messages after the read cursor. Presence comes from device heartbeats; send presence only to those who are watching. Partition messages by conversation id.

### Exercise: Order and safe retries

Write a class `Conversation(conversation_id, members)` with `send(sender, text, client_id, now)`: a sender who isn't a member raises `PermissionError("not a member")`; if this `client_id` was already used **in this conversation**, return the original message's sequence number without storing anything; otherwise store `{"seq": N, "sender", "text", "at": now}` with the next sequence number (from 1) and return N. Then `history(before=None, limit=50)`: the messages with `seq` below `before` (all, if `None`), the newest `limit` of them, returned **oldest first**; and `after(seq)`: every message with a higher sequence number, oldest first.

In [ ]:
class Conversation:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_order():\n    \"sequence numbers, not clocks\"\n    c = Conversation(\"c1\", {\"ada\", \"bob\"})\n    assert c.send(\"ada\", \"Are you free at 3?\", \"m-1\", 1000.0) == 1\n    assert c.send(\"bob\", \"Yes\", \"m-2\", 999.2) == 2\n    assert [m[\"text\"] for m in c.history()] == [\"Are you free at 3?\", \"Yes\"]\n    assert c.send(\"ada\", \"Are you free at 3?\", \"m-1\", 1001.0) == 1 and len(c.history()) == 2\n    try:\n        c.send(\"eve\", \"hi\", \"m-3\", 1002)\n    except PermissionError as e:\n        assert str(e) == \"not a member\"\n    for i in range(10):\n        c.send(\"ada\", f\"n{i}\", f\"x{i}\", 1100 + i)\n    assert [m[\"seq\"] for m in c.history(limit=3)] == [10, 11, 12]\n    assert [m[\"seq\"] for m in c.history(before=10, limit=3)] == [7, 8, 9]\n    assert [m[\"seq\"] for m in c.after(10)] == [11, 12] and c.after(12) == []")

### Exercise: Catching up

Using `Conversation` (given), write `sync(conversations, user, cursors)`: `conversations` maps ids to `Conversation` objects; `cursors` maps conversation ids to the highest sequence number the device has. For every conversation the user is a member of (in id order), return the messages after the cursor (0 for conversations not in `cursors`), as a dict from conversation id to that list, leaving out conversations with nothing new, plus the new cursors: `(updates, new_cursors)`, where `new_cursors` has an entry for every conversation the user belongs to (the last sequence number received, or the old cursor if nothing was new).

In [ ]:
class Conversation:
    def __init__(self, conversation_id, members):
        self.id, self.members, self.messages, self.client_ids = conversation_id, set(members), [], {}
    def send(self, sender, text, client_id, now):
        if client_id in self.client_ids:
            return self.client_ids[client_id]
        seq = len(self.messages) + 1
        self.messages.append({"seq": seq, "sender": sender, "text": text, "at": now})
        self.client_ids[client_id] = seq
        return seq
    def after(self, seq):
        return [m for m in self.messages if m["seq"] > seq]

def sync(conversations, user, cursors):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sync():\n    \"nothing missed, nothing repeated\"\n    a = Conversation(\"a\", {\"ada\", \"bob\"})\n    b = Conversation(\"b\", {\"ada\", \"cy\", \"dan\"})\n    c = Conversation(\"c\", {\"bob\", \"cy\"})\n    for i in range(3):\n        a.send(\"bob\", f\"a{i}\", f\"a{i}\", i)\n    b.send(\"cy\", \"b0\", \"b0\", 0)\n    c.send(\"cy\", \"c0\", \"c0\", 0)\n    updates, cursors = sync({\"a\": a, \"b\": b, \"c\": c}, \"ada\", {\"a\": 1})\n    assert {k: [m[\"text\"] for m in v] for k, v in updates.items()} == {\"a\": [\"a1\", \"a2\"], \"b\": [\"b0\"]} and cursors == {\"a\": 3, \"b\": 1}\n    updates, cursors = sync({\"a\": a, \"b\": b, \"c\": c}, \"ada\", cursors)\n    assert updates == {} and cursors == {\"a\": 3, \"b\": 1}\n    b.send(\"dan\", \"b1\", \"b1\", 5)\n    assert sync({\"a\": a, \"b\": b}, \"ada\", cursors)[0][\"b\"][0][\"text\"] == \"b1\"")

### Exercise: Ticks and unread counts

Write a class `Receipts(members)` that tracks, for each member, a `delivered` and a `read` cursor (both starting at 0; a member's read cursor also counts as delivered). `delivered(member, seq)` and `read(member, seq)` raise the cursors (never lower them). Then `status(seq, sender)`: `"read"` if every member **other than the sender** has read up to at least `seq`, else `"delivered"` if they've all had it delivered, else `"sent"`. And `unread(member, messages)`: how many messages (dicts with `seq` and `sender`) after the member's read cursor were sent by **someone else**.

In [ ]:
class Receipts:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_receipts():\n    \"cursors, not flags\"\n    r = Receipts({\"ada\", \"bob\", \"cy\"})\n    msgs = [{\"seq\": 1, \"sender\": \"ada\"}, {\"seq\": 2, \"sender\": \"bob\"}, {\"seq\": 3, \"sender\": \"ada\"}, {\"seq\": 4, \"sender\": \"cy\"}]\n    assert r.status(1, \"ada\") == \"sent\" and r.unread(\"bob\", msgs) == 3 and r.unread(\"ada\", msgs) == 2\n    r.delivered(\"bob\", 3)\n    r.delivered(\"cy\", 4)\n    assert r.status(3, \"ada\") == \"delivered\" and r.status(4, \"cy\") == \"sent\"\n    r.read(\"bob\", 4)\n    r.delivered(\"bob\", 2)\n    assert r.receipts[\"bob\"] == {\"delivered\": 4, \"read\": 4}\n    r.read(\"cy\", 3)\n    assert r.status(3, \"ada\") == \"read\" and r.status(4, \"ada\") == \"delivered\"\n    assert r.unread(\"cy\", msgs) == 0 and r.unread(\"ada\", msgs) == 2")

### Exercise: Online, away, offline

Write a class `Presence(clock, online_for=60, away_for=300)` with `heartbeat(user, device)` (record now for that device) and `state(user)`: `"online"` if any of the user's devices was heard from within `online_for` seconds (inclusive), `"away"` within `away_for` seconds, otherwise `"offline"` (including users never seen). Then `last_seen(user)`: for an offline user, a phrase from the most recent heartbeat of any device: `"last seen N minutes ago"` (whole minutes, rounded down) under an hour, `"last seen N hours ago"` under a day, else `"last seen N days ago"`; `None` if the user isn't offline or was never seen.

In [ ]:
class Presence:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_presence():\n    \"any device counts\"\n    class Clock:\n        t = 0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    p = Presence(clock)\n    p.heartbeat(\"ada\", \"phone\")\n    clock.t = 50\n    p.heartbeat(\"ada\", \"laptop\")\n    clock.t = 110\n    assert p.state(\"ada\") == \"online\" and p.last_seen(\"ada\") is None\n    clock.t = 111\n    assert p.state(\"ada\") == \"away\"\n    clock.t = 50 + 301\n    assert p.state(\"ada\") == \"offline\" and p.last_seen(\"ada\") == \"last seen 5 minutes ago\"\n    clock.t = 50 + 7300\n    assert p.last_seen(\"ada\") == \"last seen 2 hours ago\"\n    clock.t = 50 + 3 * 86400\n    assert p.last_seen(\"ada\") == \"last seen 3 days ago\" and p.state(\"ghost\") == \"offline\" and p.last_seen(\"ghost\") is None")

### Exercise: Routing to devices

Write a class `Router` with a session registry: `connect(user, device, gateway)` and `disconnect(user, device)`. `fan_out(message, members, sender_device=None)` returns `(deliveries, pushes)`: `deliveries` is a sorted list of `(gateway, user, device)` for every connected device of every member, **except** the device that sent the message; `pushes` is the sorted list of members with no connected device at all (they'll get a push notification). A member whose only connected device is the sending device gets neither.

In [ ]:
class Router:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_routing():\n    \"every device, push for the rest\"\n    r = Router()\n    r.connect(\"ada\", \"phone\", \"gw-1\")\n    r.connect(\"ada\", \"laptop\", \"gw-7\")\n    r.connect(\"bob\", \"phone\", \"gw-3\")\n    deliveries, pushes = r.fan_out({\"seq\": 1}, [\"ada\", \"bob\", \"cy\"], sender_device=(\"ada\", \"phone\"))\n    assert deliveries == [(\"gw-3\", \"bob\", \"phone\"), (\"gw-7\", \"ada\", \"laptop\")] and pushes == [\"cy\"]\n    r.disconnect(\"bob\", \"phone\")\n    r.disconnect(\"ada\", \"laptop\")\n    deliveries, pushes = r.fan_out({\"seq\": 2}, [\"ada\", \"bob\"], sender_device=(\"ada\", \"phone\"))\n    assert deliveries == [] and pushes == [\"bob\"]\n    r.disconnect(\"nobody\", \"x\")")

### Exercise: Sizing the system

Write `chat_estimate(dau, messages_per_user, bytes_per_message, peak_factor, connections_per_gateway, online_share)` returning a dict: `avg_mps` and `peak_mps` (messages per second, rounded to whole numbers), `gb_per_day` (rounded to 1 place, 10^9 bytes), `tb_per_year` (rounded to 1 place, 10^12 bytes) and `gateways` (the number of connection servers for `dau × online_share` simultaneous connections at peak, rounded up, at 70% of each gateway's capacity). Then `hot_conversations(members_by_conversation, limit)`: the sorted ids of conversations with more than `limit` members, which would make one message fan out to too many devices (and might need special handling, like Saturday's celebrities).

In [ ]:
import math

def chat_estimate(dau, messages_per_user, bytes_per_message, peak_factor, connections_per_gateway, online_share):
    ___

def hot_conversations(members_by_conversation, limit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sizing():\n    \"the numbers that shape the design\"\n    e = chat_estimate(50_000_000, 40, 200, 4, 100_000, 0.4)\n    assert e == {\"avg_mps\": 23148, \"peak_mps\": 92593, \"gb_per_day\": 400.0, \"tb_per_year\": 146.0, \"gateways\": 286}\n    assert hot_conversations({\"team\": 12, \"all-staff\": 4000, \"family\": 6, \"town\": 900}, 500) == [\"all-staff\", \"town\"]")

### Bug hunt: Answers before questions

Messages are shown in the order of the senders' device clocks, so Bob's answer comes first: this prints `['Yes', 'Are you free at 3?']`. Show them in sequence-number order, so it prints `['Are you free at 3?', 'Yes']`.

In [ ]:
messages = [
    {"seq": 1, "text": "Are you free at 3?", "client_time": 1000.0},
    {"seq": 2, "text": "Yes", "client_time": 999.2},
]
print([m["text"] for m in sorted(messages, key=lambda m: m["client_time"])])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"server sequence numbers\"\n    expect_output(\"['Are you free at 3?', 'Yes']\")")

### Bug hunt: Unread messages you wrote yourself

The unread count includes Ada's **own** messages, so after she sends two replies her badge goes up: `unread for ada: 3`. Count only messages from other people after her read cursor, so it prints `unread for ada: 1`.

In [ ]:
read_cursor = {"ada": 4}
messages = [{"seq": 5, "sender": "bob"}, {"seq": 6, "sender": "ada"}, {"seq": 7, "sender": "ada"}]

def unread(user):
    return sum(1 for m in messages if m["seq"] > read_cursor[user])

print("unread for ada:", unread("ada"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"other people's messages\"\n    expect_output(\"unread for ada: 1\")")

### Bug hunt: A message sent twice

The phone lost signal and retried, and the server stored the retry as a new message: `['On my way', 'On my way']`. Use the client message id to recognise the retry and return the original sequence number, so it prints `['On my way']`.

In [ ]:
messages, seen = [], {}

def send(text, client_id):
    messages.append(text)
    return len(messages)

send("On my way", "m-17")
send("On my way", "m-17")     # the retry
print(messages)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"idempotent sends\"\n    expect_output(\"['On my way']\")")

### Puzzle: Handling one message

Arrange the chat service's handling of an incoming message: check membership, assign a sequence number idempotently, store it before anything else, then fan out. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # what orders a conversation's messages: "device clocks" or "sequence numbers"
q = ___   # a device's cursor after receiving messages 1 to 7 of a conversation
r = ___   # "delivered" or "read": the status when every other member has it but not everyone has read it
s = ___   # the protocol gateways use so servers can push to devices
t = ___   # messages per second for 2 × 10**9 messages a day (to the nearest thousand)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"sequence numbers\" and q == 7 and r == \"delivered\", \"p, q and r\"\n    assert s in (\"WebSocket\", \"WebSockets\", \"websocket\", \"websockets\") and t == 23000, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a chat service

Write a class `ChatService(clock)` combining today's pieces (the given `Conversation`, `Receipts` and `Router`). `create(conversation_id, members)` makes a conversation with its receipts (an existing id raises `ValueError("conversation exists")`). `connect(user, device, gateway)` and `disconnect(user, device)` manage sessions. `send(conversation_id, user, device, text, client_id)` sends the message (membership and idempotency as in `Conversation`), and **for a new message only**: fans it out with `Router.fan_out` (excluding the sending device), appends each delivery to `self.outbox` as `(gateway, user, device, conversation_id, seq)`, marks it delivered for each user it reached (and the sender, who has obviously got it, as **read** up to it), appends each push to `self.pushes` as `(user, conversation_id, seq)`; return the sequence number. `sync(user, cursors)` returns what `sync` from earlier returns (given), and marks every message it returns as delivered for that user. `mark_read(conversation_id, user, seq)` records reading. `status(conversation_id, seq)` gives the message's tick (`"sent"`, `"delivered"` or `"read"`) using its sender. `unread(user)`: a dict from conversation id to the user's unread count, for every conversation they belong to, sorted by id.

In [ ]:
class Conversation:
    def __init__(self, conversation_id, members):
        self.id, self.members, self.messages, self.client_ids = conversation_id, set(members), [], {}
    def send(self, sender, text, client_id, now):
        if sender not in self.members:
            raise PermissionError("not a member")
        if client_id in self.client_ids:
            return self.client_ids[client_id]
        seq = len(self.messages) + 1
        self.messages.append({"seq": seq, "sender": sender, "text": text, "at": now})
        self.client_ids[client_id] = seq
        return seq
    def after(self, seq):
        return [m for m in self.messages if m["seq"] > seq]

class Receipts:
    def __init__(self, members):
        self.receipts = {m: {"delivered": 0, "read": 0} for m in members}
    def delivered(self, member, seq):
        self.receipts[member]["delivered"] = max(self.receipts[member]["delivered"], seq)
    def read(self, member, seq):
        r = self.receipts[member]
        r["read"] = max(r["read"], seq)
        r["delivered"] = max(r["delivered"], r["read"])
    def status(self, seq, sender):
        others = [r for m, r in self.receipts.items() if m != sender]
        if all(r["read"] >= seq for r in others):
            return "read"
        if all(r["delivered"] >= seq for r in others):
            return "delivered"
        return "sent"
    def unread(self, member, messages):
        return sum(1 for m in messages if m["seq"] > self.receipts[member]["read"] and m["sender"] != member)

class Router:
    def __init__(self):
        self.sessions = {}
    def connect(self, user, device, gateway):
        self.sessions.setdefault(user, {})[device] = gateway
    def disconnect(self, user, device):
        self.sessions.get(user, {}).pop(device, None)
    def fan_out(self, message, members, sender_device=None):
        deliveries, pushes = [], []
        for user in members:
            devices = self.sessions.get(user, {})
            if not devices:
                pushes.append(user)
                continue
            for device, gateway in devices.items():
                if (user, device) != sender_device:
                    deliveries.append((gateway, user, device))
        return sorted(deliveries), sorted(pushes)

def sync(conversations, user, cursors):
    updates, new_cursors = {}, {}
    for cid in sorted(conversations):
        conv = conversations[cid]
        if user not in conv.members:
            continue
        cursor = cursors.get(cid, 0)
        fresh = conv.after(cursor)
        if fresh:
            updates[cid] = fresh
            cursor = fresh[-1]["seq"]
        new_cursors[cid] = cursor
    return updates, new_cursors

class ChatService:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _service():\n    class Clock:\n        t = 1000\n        def __call__(self):\n            return self.t\n    s = ChatService(Clock())\n    s.create(\"team\", [\"ada\", \"bob\", \"cy\"])\n    s.create(\"dm\", [\"ada\", \"bob\"])\n    s.connect(\"ada\", \"phone\", \"gw-1\")\n    s.connect(\"ada\", \"laptop\", \"gw-2\")\n    s.connect(\"bob\", \"phone\", \"gw-3\")\n    return s\n\ndef test_send_and_fan_out():\n    \"devices, pushes and ticks\"\n    s = _service()\n    assert s.send(\"team\", \"ada\", \"phone\", \"Stand-up at 10\", \"m1\") == 1\n    assert s.outbox == [(\"gw-2\", \"ada\", \"laptop\", \"team\", 1), (\"gw-3\", \"bob\", \"phone\", \"team\", 1)] and s.pushes == [(\"cy\", \"team\", 1)]\n    assert s.status(\"team\", 1) == \"sent\"\n    assert s.send(\"team\", \"ada\", \"phone\", \"Stand-up at 10\", \"m1\") == 1 and len(s.outbox) == 2, \"a retry fans out nothing\"\n    updates, cursors = s.sync(\"cy\", {})\n    assert [m[\"text\"] for m in updates[\"team\"]] == [\"Stand-up at 10\"] and cursors == {\"team\": 1}\n    assert s.status(\"team\", 1) == \"delivered\"\n    s.mark_read(\"team\", \"bob\", 1)\n    s.mark_read(\"team\", \"cy\", 1)\n    assert s.status(\"team\", 1) == \"read\"\n    try:\n        s.create(\"team\", [\"x\"])\n    except ValueError as e:\n        assert str(e) == \"conversation exists\"\n\ndef test_unread():\n    \"badges per conversation\"\n    s = _service()\n    s.send(\"team\", \"bob\", \"phone\", \"one\", \"b1\")\n    s.send(\"team\", \"cy\", \"web\", \"two\", \"c1\")\n    s.send(\"dm\", \"bob\", \"phone\", \"psst\", \"b2\")\n    assert s.unread(\"ada\") == {\"dm\": 1, \"team\": 2} and s.unread(\"cy\") == {\"team\": 0}\n    s.send(\"team\", \"ada\", \"laptop\", \"reply\", \"a1\")\n    assert s.unread(\"ada\") == {\"dm\": 1, \"team\": 0}, \"sending marks the conversation read\"\n    assert s.unread(\"cy\") == {\"team\": 1}\n    s.mark_read(\"team\", \"cy\", 3)\n    assert s.unread(\"cy\") == {\"team\": 0}\n    try:\n        s.send(\"dm\", \"cy\", \"web\", \"hi\", \"c2\")\n    except PermissionError:\n        pass\n    else:\n        raise AssertionError(\"cy isn't in the dm\")")

## Warm-up

Ordering a conversation by the senders' clocks puts Bob's answer before Ada's question. Monday's lesson again: clocks on different devices can't order events. A chat server gives each message a **sequence number** within its conversation as it arrives, and that number, not any clock, is the order everyone sees.

## Clarify and estimate

**Questions worth asking**: one-to-one and groups? (Both; groups up to 500.) Multiple devices per person? (Yes: phone, laptop, tablet.) Message history kept? (Forever.) Media? (Out of scope today: attachments go to object storage, and messages carry links.) Receipts and presence? (Yes.) End-to-end encryption? (Not today; mention it changes what the server can do.)

**Requirements**:

- send and receive messages in real time; **the same order for everyone** in a conversation;
- **no message lost**, none shown twice, even when phones lose signal mid-send;
- devices that were offline catch up on everything they missed;
- delivered and read receipts, unread counts, online status and "last seen".

**Estimates**: 50 million daily users sending 40 messages a day is `2 × 10^9` messages a day, about **23,000 a second** on average and perhaps 100,000 at peak. At 200 bytes each that's **400 GB a day**, about 150 TB a year, before replication: far beyond one database, so storage must be **partitioned**. With 50 million users online at peak and each device holding a connection, at around 100,000 connections per server that's **hundreds of connection servers**.

## The high-level design

```
devices ──WebSocket── gateways ──► chat service ──► message store (partitioned by conversation)
                         ▲               │
                         │               ├──► session registry: which gateway holds each device
                         └──── fan-out ──┤
                                         └──► push notification service (for offline devices)
```

- **Gateways** (connection servers) hold long-lived **WebSocket** connections (one per device), so the server can push messages without the client asking. They do little else, so each can hold a huge number of connections.
- A **session registry** (a fast shared store, Week 81) records which gateway each **device** is connected to.
- The **chat service** receives a message, assigns its sequence number, **stores it first**, and only then fans it out: for each member's devices, the registry says which gateway to forward it to; devices that aren't connected get a **push notification** instead.
- The **message store** is partitioned by **conversation id** (Week 81): a conversation's messages are together and in order, and each conversation's sequence counter lives with them.

## Deep dive: order and sync

**Sequence numbers.** Each conversation has a counter; the chat service gives each message the next number atomically (Week 82's rate-limiter lesson: increment, then use the returned value). Clients display messages by sequence number. Because the counter is per conversation, conversations never wait for each other.

**Safe retries.** A phone sends a message, the signal drops, and it doesn't know whether the server received it, so it sends again. To stop the message appearing twice, the client gives each message a **client message id** (a random id it generates once and reuses for every retry); the server keeps the ids it has seen per conversation and, for a repeat, returns the original message's sequence number instead of storing a new one: Week 77's idempotency keys again.

**Sync.** Each device remembers, for each conversation, the **highest sequence number it has received**: its **cursor**. On reconnecting, it sends its cursors, and the server returns every message after each one (and the cursor to use next). No message is missed, because sequence numbers have no gaps, and nothing is sent twice. New conversations the device hasn't seen start from cursor 0.

## Deep dive: receipts and presence

**Receipts** are cursors too. For each member of a conversation, the server keeps two numbers: the highest sequence **delivered** to any of their devices, and the highest they've **read**. The sender's "delivered" tick for message 41 appears once every other member's delivered number is at least 41; "read by everyone" once every other member's read number is. A member's **unread count** is the number of messages after their read number that **other people** sent. All of this is a handful of integers per member, not a flag per message.

**Presence** comes from heartbeats (Monday's failure detector): each device pings every 30 seconds or so; a user is **online** if any of their devices was heard from recently, **away** if only a little while ago, and otherwise **offline** with a "last seen" time. Presence changes are sent to people who have the user's chat open, not to everyone who knows them: in a large system, presence fan-out can easily cost more than the messages themselves.

## Practice

## Quiz

## Daily challenge

## Before you go

Write a one-page design for a variation (a chat system with end-to-end encryption, say) and list which of today's server-side features would have to change because the server can no longer read the messages. Tomorrow is the Week 82 **Forge**: a full design, built and tested, for a system where many people compete for the same scarce thing.